# XGBoost

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import math
import time
import warnings
from datetime import timedelta
from tabulate import tabulate
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import f1_score, make_scorer, classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight 

# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')


"""
    

"""


# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training


In [6]:
def training(file_path, csv_name):
   
    df = pd.read_csv(file_path)
    
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]', 'HER2 [SII]']
    
    df_validi = df.dropna(subset=original_target_list).copy()
    
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    df_validi['HER2_class'] = (df_validi['HER2 [SII]'] >= 3).astype(int)
    
    final_target_list = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']
    
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']

    # XGBoost gestisce NaN, ma riempire è più sicuro per coerenza
    features = features.fillna(features.mean())
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]
    
    cv = GroupKFold(n_splits=5)

    # Configurazione Base
    base_model = XGBClassifier(
        random_state=42,
        n_jobs=1,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist'  # Più veloce ed efficiente
    )
    
    multi_output_model = MultiOutputClassifier(base_model)

    # Iperparametri Ottimizzati per Dataset Medici (Piccoli & Rumorosi)
    iperparametri = {
      'estimator__n_estimators': [50, 100],        # Pochi alberi per evitare overfitting
      'estimator__max_depth': [2, 3],              # Alberi bassi (stump) per generalizzare meglio
      'estimator__learning_rate': [0.05, 0.1],     
      'estimator__min_child_weight': [2, 4],       # Aumentato per richiedere più campioni per foglia
      'estimator__reg_alpha': [0, 0.1, 1],         # Regolarizzazione L1 (importante per feature selection)
      'estimator__subsample': [0.8],               # Previene overfitting campionando righe
      'estimator__colsample_bytree': [0.8]         # Previene overfitting campionando colonne
    }
    
    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0))
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    # Calcolo pesi campioni "globali" per gestire lo sbilanciamento medio
    sample_weights = compute_sample_weight(class_weight='balanced', y=target['HER2_class']) # Uso HER2 come riferimento critico

    total_combinations = math.prod(len(v) for v in iperparametri.values())
    print(f"\nInizio Grid Search ({total_combinations} combinazioni) per: {csv_name}")

    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=cv,                  
        scoring=scorer,
        n_jobs=-1,              
        verbose=1,
        return_train_score=False 
    )
    
    # Passo i sample_weights per aiutare col bilanciamento (supportato da MultiOutputClassifier fit param routing)
    grid_search.fit(features, target, groups=groups) # sample_weight non supportato facilmente in GridCV MultiOutput

    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    fold_reports = []
    
    clean_best_params = {k.replace('estimator__', ''): v for k, v in best_params.items()}
   
    final_model_params = {
        'random_state': 42,
        'n_jobs': 1,
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        **clean_best_params
    }
    
    for train_idx, test_idx in cv.split(features, target, groups):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        # Calcolo pesi specifici per il fold di training (basato su HER2 o target più raro)
        fold_weights = compute_sample_weight('balanced', y_train['HER2_class'])

        model_clone = MultiOutputClassifier(XGBClassifier(**final_model_params))
        # Qui posso passare sample_weight perché chiamo fit direttamente
        model_clone.fit(X_train, y_train, sample_weight=fold_weights)

        y_pred = model_clone.predict(X_test)
        y_proba_list = model_clone.predict_proba(X_test)

        report_dict = {}
        for i, col in enumerate(final_target_list):
            rep = classification_report(
                y_test.iloc[:, i],
                y_pred[:, i],
                output_dict=True,
                zero_division=0
            )
            
            unique_classes = np.unique(y_test.iloc[:, i])

            if len(unique_classes) < 2:
                auc_val = np.nan 
            else:
                try:
                    if y_proba_list[i].shape[1] == 2:
                        auc_val = roc_auc_score(y_test.iloc[:, i], y_proba_list[i][:, 1])
                    else:
                        auc_val = 0.5 
                except ValueError:
                    auc_val = np.nan
            
            rep['auc'] = auc_val
            report_dict[col] = rep

        fold_reports.append(report_dict)

    final_result = [{
        **clean_best_params,
        'mean_score': best_score,
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports
    }]

    return final_result

# Vado a stampare il risultato in un formato leggibile

In [7]:
def print_grid_search_results(results_per_dataset):

    print("\n" + "=" * 80)
    print(" " * 25 + "RIEPILOGO DEI MIGLIORI RISULTATI (XGBoost)")
    print("=" * 80)

    summary_data = []

    for name, metrics_list in results_per_dataset.items():
        best_result = metrics_list[0]

        auc_values = []
        if best_result.get('fold_reports'):
            for fold_rep in best_result['fold_reports']:
                for target_metrics in fold_rep.values():
                    if isinstance(target_metrics, dict) and 'auc' in target_metrics:
                        auc_values.append(target_metrics['auc'])

        mean_auc = np.nanmean(auc_values) if auc_values else 0.0

        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}")
        print(f" Mean AUC    = {mean_auc:.3f}\n")

        print("Iperparametri Ottimali:")
        params_to_show = [
            ('N Estimators', 'n_estimators'),
            ('Max Depth', 'max_depth'),
            ('Learning Rate', 'learning_rate'),
            ('Min Child Weight', 'min_child_weight'),
            ('Reg Alpha (L1)', 'reg_alpha'),    # Aggiunto alla visualizzazione
            ('Subsample', 'subsample')
        ]

        params_table = []
        for label, key in params_to_show:
            val = best_result.get(key)
            if val is not None:
                params_table.append([label, val])

        print(tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))

        print("\n Metriche di Classificazione per Target (Dettaglio primo fold):\n")
        target_names = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

        if best_result['fold_reports']:
            first_fold_report = best_result['fold_reports'][0]

            for target_name in target_names:
                if target_name not in first_fold_report: continue

                current_target_report = first_fold_report[target_name]
                rows = []
                classes = [c for c in current_target_report.keys() if c in ['0', '1']]

                for cls in classes:
                    metrics = current_target_report[cls]
                    rows.append([
                        f"Classe {cls}",
                        f"{metrics['precision']:.3f}",
                        f"{metrics['recall']:.3f}",
                        f"{metrics['f1-score']:.3f}",
                        int(metrics['support'])
                    ])

                auc_val = current_target_report.get('auc')
                auc_str = f"  ---> AUC: {auc_val:.3f}" if (auc_val is not None and not np.isnan(auc_val)) else ""

                print(f"  Target: {target_name} {auc_str}")
                print(tabulate(rows, headers=['', 'Precision', 'Recall', 'F1-score', 'Support'],
                             tablefmt='simple', colalign=('left', 'center', 'center', 'center', 'center')))
                print()
        else:
            print("Nessun report dettagliato disponibile.")

        summary_data.append([
            name,
            f"{best_result['mean_score']:.3f}",
            f"{best_result['std_score']:.3f}",
            f"{mean_auc:.3f}", 
            best_result.get('max_depth'),
            best_result.get('learning_rate'),
            best_result.get('n_estimators')
        ])

    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")

    summary_data.sort(key=lambda x: float(x[1]), reverse=True)

    print(tabulate(summary_data,
                   headers=['Dataset', 'F1-score', 'Std Dev', 'AUC', 'Depth', 'LR', 'N Est.'],
                   tablefmt='grid',
                   floatfmt=('.3f', '.3f', '.3f', '.3f', '.0f', '.0f', '.0f')))

# Lettura dei file

In [8]:
start_time = time.time()


# Eseguo il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)




end_time = time.time()
# Calcolo il tempo impiegato
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



Inizio Grid Search (48 combinazioni) per: t2_medsam
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Inizio Grid Search (48 combinazioni) per: t2_preprocessed
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Inizio Grid Search (48 combinazioni) per: t2_original
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Inizio Grid Search (48 combinazioni) per: medsam_dynamic
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Inizio Grid Search (48 combinazioni) per: preprocessed_dynamic
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Inizio Grid Search (48 combinazioni) per: original_dynamic
Fitting 5 folds for each of 48 candidates, totalling 240 fits

                         RIEPILOGO DEI MIGLIORI RISULTATI (XGBoost)

────────────────────────────────────────────────────────────────────────────────
 Dataset: t2_medsam
────────────────────────────────────────────────────────────────────────────────

 Performance: F1-score = 0